<a href="https://colab.research.google.com/github/VJN1089/ScamShield-Bharat_/blob/main/ScamShield_BharatFinal.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [32]:
!pip install -q streamlit easyocr pillow numpy requests

In [33]:
import os

PROJECT = "/content/ScamShield_Bharat"
os.makedirs(PROJECT, exist_ok=True)

print("✅ Project folder created:", PROJECT)

✅ Project folder created: /content/ScamShield_Bharat


In [34]:
%%writefile /content/ScamShield_Bharat/detector.py

import re

RED_FLAGS = {
    "Guaranteed Returns": [
        "guaranteed return",
        "guaranteed profit",
        "fixed return",
        "assured return",
        "double your money",
        "100% profit",
        "40% return",
        "50% return"
    ],

    "Urgency": [
        "act now",
        "today only",
        "limited time",
        "urgent",
        "last chance",
        "expires today",
        "only 3 slots",
        "only 5 slots"
    ],

    "Tip Group": [
        "telegram",
        "whatsapp group",
        "vip group",
        "signal group",
        "investment group",
        "trading group",
        "premium group",
        "tip group"
    ],

    "Pump and Dump Language": [
        "pump",
        "dump",
        "buy before everyone",
        "secret stock tip",
        "rocket stock",
        "multibagger guaranteed",
        "operator buying",
        "insider tip"
    ],

    "SEBI / Authority Claim": [
        "sebi approved",
        "sebi registered",
        "sebi registration",
        "sebi license",
        "sebi certified",
        "rbi approved",
        "government approved",
        "official sebi"
    ],

    "Remote Access": [
        "anydesk",
        "teamviewer",
        "quicksupport",
        "remote access",
        "remote desktop",
        "screen sharing",
        "give remote access"
    ],

    "Unknown App": [
        "download apk",
        "install apk",
        "trading apk",
        "download this app",
        "install this app",
        "unknown app"
    ],

    "Withdrawal Fee": [
        "withdrawal fee",
        "pay tax",
        "processing fee",
        "unlock withdrawal",
        "withdrawal tax",
        "pay a fee",
        "release your profit"
    ],

    "Payment Request": [
        "deposit",
        "transfer money",
        "send money",
        "upi payment",
        "send payment",
        "pay now",
        "make payment"
    ]
}


def clean_text(text):
    text = str(text).lower()
    text = re.sub(r"\s+", " ", text)
    return text.strip()


def analyze_text(text):

    text = clean_text(text)

    flags = []
    keywords = []

    for category, words in RED_FLAGS.items():

        for word in words:

            if word in text:

                flags.append(category)
                keywords.append(word)

                break

    score = min(len(flags), 10)

    if score >= 5:
        level = "HIGH CONCERN"

    elif score >= 3:
        level = "CAUTION"

    else:
        level = "LOW CONCERN"

    if score == 0:

        explanation = (
            "No known warning pattern was detected. "
            "This does not prove that the message is safe."
        )

    else:

        explanation = (
            "The message contains warning patterns associated "
            "with possible investment scam tactics."
        )

    actions = [
        "Pause before transferring money.",
        "Verify the person or organization independently.",
        "Do not install unknown APK or remote-access applications.",
        "Never share OTP, PIN or password.",
        "Be cautious about guaranteed or unusually high returns.",
        "Verify important claims using official sources."
    ]

    return {
        "level": level,
        "score": score,
        "flags": flags,
        "keywords": keywords,
        "explanation": explanation,
        "actions": actions
    }

Overwriting /content/ScamShield_Bharat/detector.py


In [35]:
from sys import path

path.append("/content/ScamShield_Bharat")

from detector import analyze_text

test_message = """
SEBI approved VIP Telegram investment group.

Guaranteed 40% return today only.

Deposit ₹10,000 immediately.

Download our trading APK.

Use AnyDesk for account verification.

Pay a withdrawal fee to release your profit.
"""

result = analyze_text(test_message)

print("=" * 50)
print("🛡️ SCAMSHIELD BHARAT TEST")
print("=" * 50)

print("\nRisk Level:", result["level"])
print("Risk Indicator:", str(result["score"]) + "/10")

print("\nWarning Signals:")

for flag in result["flags"]:
    print("🚩", flag)

print("\nMatched Keywords:")

for keyword in result["keywords"]:
    print("•", keyword)

print("\nWhy this matters:")
print(result["explanation"])

print("\nSafer Next Steps:")

for action in result["actions"]:
    print("✓", action)

🛡️ SCAMSHIELD BHARAT TEST

Risk Level: HIGH CONCERN
Risk Indicator: 7/10

Warning Signals:
🚩 Urgency
🚩 Tip Group
🚩 SEBI / Authority Claim
🚩 Remote Access
🚩 Unknown App
🚩 Withdrawal Blocking / Fee
🚩 Payment Request

Matched Keywords:
• today only
• telegram
• sebi approved
• anydesk
• trading apk
• withdrawal fee
• deposit

Why this matters:
The message contains warning patterns commonly associated with financial scam attempts, such as pressure tactics, payment requests, unrealistic returns, authority claims, investment groups, remote access requests or withdrawal fees.

Safer Next Steps:
✓ Pause before transferring money.
✓ Verify the person or organization independently.
✓ Do not install unknown APK or remote-access applications.
✓ Never share OTP, PIN or password.
✓ Be cautious about guaranteed or unusually high returns.
✓ Verify important claims using official sources.


In [36]:
%%writefile /content/ScamShield_Bharat/app.py

import streamlit as st
import re

from urllib.parse import urlparse

from detector import analyze_text


st.set_page_config(
    page_title="ScamShield Bharat",
    page_icon="🛡️",
    layout="centered"
)


st.title("🛡️ ScamShield Bharat")

st.subheader("Pause. Verify. Protect.")

st.write(
    "An explainable digital safety assistant for "
    "suspicious investment messages, tip groups and links."
)


language = st.selectbox(
    "🌐 Language",
    ["English", "Hindi", "Kannada"]
)


tab1, tab2, tab3, tab4 = st.tabs([
    "💬 Message Scanner",
    "📷 Tip-Group Profiler",
    "🔗 Link Checker",
    "🛡️ Safety Guide"
])


# ============================================================
# MESSAGE SCANNER
# ============================================================

with tab1:

    st.markdown("### 💬 Scam & Claim Verifier")

    message = st.text_area(
        "Paste suspicious investment message",
        height=200,
        placeholder=(
            "Example: Guaranteed 40% return. "
            "Join our Telegram group and deposit today..."
        )
    )

    if st.button("🔍 Analyze Message"):

        if not message.strip():

            st.warning("Please enter a message.")

        else:

            result = analyze_text(message)

            if result["level"] == "HIGH CONCERN":

                st.error("🚨 " + result["level"])

            elif result["level"] == "CAUTION":

                st.warning("🟡 " + result["level"])

            else:

                st.success("🟢 " + result["level"])


            st.metric(
                "Risk Indicator",
                f"{result['score']}/10"
            )


            st.markdown("### 🚩 Detected Warning Signals")

            for flag in result["flags"]:

                st.write("🚩 " + flag)


            st.markdown("### 🔎 Why this matters")

            st.write(result["explanation"])


            st.markdown("### 🛡️ Safer Next Steps")

            for action in result["actions"]:

                st.write("✓ " + action)


# ============================================================
# SCREENSHOT PROFILER
# ============================================================

with tab2:

    st.markdown(
        "### 📷 WhatsApp / Telegram Tip-Group Profiler"
    )

    st.write(
        "Upload a screenshot of a suspicious investment "
        "message or tip group."
    )


    uploaded_image = st.file_uploader(
        "Upload Screenshot",
        type=["png", "jpg", "jpeg"]
    )


    if uploaded_image:

        st.image(
            uploaded_image,
            caption="Uploaded Screenshot",
            use_container_width=True
        )


        if st.button("🔍 Analyze Screenshot"):

            try:

                import easyocr
                import numpy as np
                from PIL import Image

                image = Image.open(uploaded_image)

                image_array = np.array(image)

                st.info("Reading screenshot...")

                reader = easyocr.Reader(
                    ["en"],
                    gpu=False
                )

                results = reader.readtext(
                    image_array
                )


                extracted_text = "\n".join(
                    item[1]
                    for item in results
                )


                if not extracted_text.strip():

                    st.warning(
                        "No readable text detected."
                    )

                else:

                    st.markdown(
                        "### 📄 Extracted Text"
                    )

                    st.text_area(
                        "OCR Result",
                        extracted_text,
                        height=180
                    )


                    result = analyze_text(
                        extracted_text
                    )


                    if result["level"] == "HIGH CONCERN":

                        st.error(
                            "🚨 " +
                            result["level"]
                        )

                    elif result["level"] == "CAUTION":

                        st.warning(
                            "🟡 " +
                            result["level"]
                        )

                    else:

                        st.success(
                            "🟢 " +
                            result["level"]
                        )


                    st.metric(
                        "Risk Indicator",
                        f"{result['score']}/10"
                    )


                    st.markdown(
                        "### 🚩 Warning Signals"
                    )

                    for flag in result["flags"]:

                        st.write(
                            "🚩 " + flag
                        )


            except Exception as e:

                st.error(
                    "Screenshot processing failed."
                )

                st.write(str(e))


# ============================================================
# LINK CHECKER
# ============================================================

with tab3:

    st.markdown(
        "### 🔗 Suspicious Link Checker"
    )

    url = st.text_input(
        "Paste suspicious investment link",
        placeholder="https://example.com/investment"
    )


    if st.button("🔎 Check Link"):

        if not url.strip():

            st.warning(
                "Please enter a URL."
            )

        else:

            if not url.startswith(
                ("http://", "https://")
            ):

                url = "https://" + url


            parsed = urlparse(url)

            domain = parsed.netloc.lower()

            warnings = []


            if parsed.scheme != "https":

                warnings.append(
                    "Website does not use HTTPS."
                )


            if re.match(
                r"^\d{1,3}(\.\d{1,3}){3}$",
                domain
            ):

                warnings.append(
                    "URL uses an IP address."
                )


            suspicious_words = [
                "guaranteed",
                "profit",
                "double",
                "bonus",
                "urgent",
                "investment"
            ]


            for word in suspicious_words:

                if word in url.lower():

                    warnings.append(
                        f"URL contains '{word}'."
                    )


            if len(warnings) >= 3:

                st.error(
                    "🚨 HIGH CONCERN"
                )

            elif len(warnings) >= 1:

                st.warning(
                    "🟡 CAUTION"
                )

            else:

                st.success(
                    "🟢 LOW CONCERN"
                )


            st.markdown(
                "### 🚩 URL Warning Indicators"
            )


            if warnings:

                for warning in warnings:

                    st.write(
                        "🚩 " + warning
                    )

            else:

                st.write(
                    "No basic URL warning indicator detected."
                )


            st.info(
                "A low indicator result does not prove "
                "that a website is safe."
            )


# ============================================================
# SAFETY GUIDE
# ============================================================

with tab4:

    st.markdown(
        "### 🛡️ Investor Safety Guide"
    )


    st.markdown("""
### 1️⃣ Pause

Do not act immediately because someone creates urgency.

### 2️⃣ Verify

Independently verify the person, organization and claim.

### 3️⃣ Protect

Never share OTP, PIN, password or banking credentials.

### 4️⃣ Avoid Remote Access

Do not give unknown people remote access to your device.

### 5️⃣ Question Guaranteed Returns

Be cautious about guaranteed or unusually high returns.

### 6️⃣ Be Careful With Tip Groups

Do not blindly trust WhatsApp or Telegram investment groups.

### 7️⃣ Don't Pay to Unlock Withdrawals

Requests for additional money to release supposed profits are a major warning signal.

### 8️⃣ Verify Official Claims

Do not rely only on a registration number or logo shown inside a message or screenshot.
""")


st.divider()

st.markdown("### 🔐 Privacy First")

st.write(
    "Do not enter OTPs, UPI PINs, passwords, "
    "card PINs or banking credentials."
)


st.caption(
    "ScamShield Bharat is a safety-assistance prototype. "
    "It identifies warning patterns but does not provide "
    "investment advice or a definitive fraud determination."
)

Overwriting /content/ScamShield_Bharat/app.py


In [37]:
!pkill -f streamlit || true

^C


In [38]:
import subprocess
import time

process = subprocess.Popen(
    [
        "streamlit",
        "run",
        "/content/ScamShield_Bharat/app.py",
        "--server.port=8501",
        "--server.address=0.0.0.0",
        "--server.headless=true",
        "--browser.gatherUsageStats=false"
    ]
)

time.sleep(8)

print("🛡️ ScamShield Bharat started")
print("PID:", process.pid)

🛡️ ScamShield Bharat started
PID: 16376


In [39]:
import requests

try:

    response = requests.get(
        "http://127.0.0.1:8501",
        timeout=10
    )

    print("✅ STREAMLIT IS RUNNING")
    print("Status:", response.status_code)

except Exception as e:

    print("❌ STREAMLIT FAILED")
    print(e)

✅ STREAMLIT IS RUNNING
Status: 200


In [40]:
from google.colab.output import eval_js

url = eval_js(
    "google.colab.kernel.proxyPort(8501)"
)

print("🛡️ SCAMSHIELD BHARAT")
print("=" * 50)
print(url)

🛡️ SCAMSHIELD BHARAT
https://8501-m-s-kkb-usw4a0-2awco47z244xq-a.us-west4-0.prod.colab.dev
